# Tube Detection & Angle Estimation — Integrated Pipeline

**Four stages, run top-to-bottom:**

| # | Stage | What happens |
|---|-------|--------------|
| 1 | **Prep** | Convert `annotations.csv` → YOLO OBB label `.txt` files; write `data.yaml` |
| 2 | **YOLO training** | Train YOLOv8n-obb to detect tube bounding boxes |
| 3 | **ResNet-18 training** | Use **ground-truth** bboxes to crop tubes, train ResNet-18 to predict angle (sin/cos regression) |
| 4 | **Full inference** | YOLO → crop → ResNet-18 → CSV in the same column format as `annotations.csv` |

**Expected directory layout before running:**
```
.
├── data/
│   ├── annotations.csv          # ground-truth: image,center_x,center_y,bbox_w,bbox_h,bbox_rotation,angle_deg
│   └── annotated_images/        # all 70 source images
├── data.yaml                    # auto-generated by Cell 1-B
├── yolov8n-obb.pt               # pre-trained YOLO weights (download once)
└── func.ipynb
```

---
## Stage 0 — Imports & Shared Utilities

In [53]:
import os
import math
import shutil
import random

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from ultralytics import YOLO

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Paths (edit these if your layout differs) ─────────────────────────────────
ANNOTATIONS_CSV  = './data/annotations.csv'
IMAGES_DIR       = './data/annotated_images'
LABELS_DIR       = './data/labels'           # YOLO OBB .txt files go here
DATA_YAML        = './data.yaml'
YOLO_PRETRAINED  = './yolov8n-obb.pt'
YOLO_WEIGHTS     = None  # resolved automatically after training (see Stage 2)
RESNET_WEIGHTS   = './resnet18_angle.pth'
OUTPUT_CSV       = './predictions.csv'

# ── Hardware ──────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

# ── Shared helper: crop a rotated bounding box from an image ──────────────────
def crop_rotated_bbox(image_path, center_x, center_y, bbox_w, bbox_h, bbox_rotation_deg):
    """
    Rotate the full image so the bbox becomes axis-aligned, then crop.
    Returns a BGR numpy array of shape (bbox_h, bbox_w, 3).
    Raises ValueError if the crop would be empty.
    """
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f'Cannot read: {image_path}')

    img_h, img_w = image.shape[:2]
    M = cv2.getRotationMatrix2D((center_x, center_y), bbox_rotation_deg, 1.0)
    rotated = cv2.warpAffine(image, M, (img_w, img_h),
                             flags=cv2.INTER_LINEAR,
                             borderMode=cv2.BORDER_REFLECT_101)

    x1 = max(0, int(center_x - bbox_w / 2))
    y1 = max(0, int(center_y - bbox_h / 2))
    x2 = min(img_w, x1 + int(bbox_w))
    y2 = min(img_h, y1 + int(bbox_h))

    crop = rotated[y1:y2, x1:x2]
    if crop.size == 0:
        raise ValueError(f'Empty crop for bbox at ({center_x},{center_y}) in {image_path}')
    return crop

Using device: cuda


---
## Stage 1-A — Convert `annotations.csv` → YOLO OBB Labels

In [54]:
def convert_csv_to_yolo_obb(csv_path, images_dir, labels_dir):
    """
    Reads annotations.csv and writes one .txt per image in YOLO OBB format:
        class_id  x1 y1 x2 y2 x3 y3 x4 y4   (all normalised to [0,1])
    One row per tube instance; file is created fresh (not appended) per image.
    """
    os.makedirs(labels_dir, exist_ok=True)

    # Clear stale label files so a re-run starts clean
    for f in os.listdir(labels_dir):
        if f.endswith('.txt'):
            os.remove(os.path.join(labels_dir, f))

    df = pd.read_csv(csv_path)
    required = {'image', 'center_x', 'center_y', 'bbox_w', 'bbox_h', 'bbox_rotation'}
    missing  = required - set(df.columns)
    if missing:
        raise ValueError(f'annotations.csv is missing columns: {missing}')

    skipped = 0
    for img_name, group in df.groupby('image'):
        img_path = os.path.join(images_dir, img_name)
        img      = cv2.imread(img_path)
        if img is None:
            print(f'  [WARN] Cannot read {img_path} — skipping')
            skipped += 1
            continue

        img_h, img_w = img.shape[:2]
        txt_name     = os.path.splitext(img_name)[0] + '.txt'
        txt_path     = os.path.join(labels_dir, txt_name)

        lines = []
        for _, row in group.iterrows():
            rect   = ((row['center_x'], row['center_y']),
                      (row['bbox_w'],   row['bbox_h']),
                       row['bbox_rotation'])
            box    = cv2.boxPoints(rect)          # shape (4, 2), absolute px
            box[:, 0] /= img_w
            box[:, 1] /= img_h
            coords = box.flatten()
            lines.append('0 ' + ' '.join(f'{c:.6f}' for c in coords))

        with open(txt_path, 'w') as fh:
            fh.write('\n'.join(lines) + '\n')

    total = df['image'].nunique() - skipped
    print(f'Conversion complete — wrote labels for {total} images '
          f'({skipped} skipped) → {labels_dir}')


convert_csv_to_yolo_obb(ANNOTATIONS_CSV, IMAGES_DIR, LABELS_DIR)

Conversion complete — wrote labels for 70 images (0 skipped) → ./data/labels


---
## Stage 1-B — Write `data.yaml` and Verify Dataset

In [55]:
import yaml

# Ultralytics resolves label paths by replacing the last 'images' path component
# with 'labels'. It also appends the split name (e.g. 'train') as a subfolder.
# The only layout that works reliably across versions is:
#   data/images/train/<image files>
#   data/labels/train/<label .txt files>
# We build that structure here using symlinks so no data is copied.

import pathlib

DATA_ROOT   = pathlib.Path('./data').resolve()
IMG_TRAIN   = DATA_ROOT / 'images' / 'train'
LBL_TRAIN   = DATA_ROOT / 'labels' / 'train'
SRC_IMAGES  = DATA_ROOT / 'annotated_images'
SRC_LABELS  = DATA_ROOT / 'labels_src'   # rename from original labels/ to avoid clash

# 1. Move generated label files into a neutral source folder so 'labels/'
#    is free for Ultralytics to own.
import shutil, os
orig_labels = DATA_ROOT / 'labels'
if orig_labels.exists() and not orig_labels.is_symlink():
    if SRC_LABELS.exists():
        shutil.rmtree(SRC_LABELS)
    orig_labels.rename(SRC_LABELS)
    # Update LABELS_DIR so Stage 1-A still writes to the right place on re-runs
    # (reassign the global used by convert_csv_to_yolo_obb)
    globals()['LABELS_DIR'] = str(SRC_LABELS)

# 2. Create the nested image and label dirs Ultralytics expects
IMG_TRAIN.mkdir(parents=True, exist_ok=True)
LBL_TRAIN.mkdir(parents=True, exist_ok=True)

# 3. Symlink every image file into images/train/
for img_file in SRC_IMAGES.iterdir():
    if img_file.suffix.lower() in ('.png', '.jpg', '.jpeg'):
        link = IMG_TRAIN / img_file.name
        if link.is_symlink():
            link.unlink()
        os.symlink(img_file, link)

# 4. Symlink every label .txt file into labels/train/
for lbl_file in SRC_LABELS.iterdir():
    if lbl_file.suffix == '.txt':
        link = LBL_TRAIN / lbl_file.name
        if link.is_symlink():
            link.unlink()
        os.symlink(lbl_file, link)

# 5. Write data.yaml with explicit absolute path so there is no ambiguity
data_cfg = {
    'path'  : str(DATA_ROOT),
    'train' : 'images/train',
    'val'   : 'images/train',   # same split; swap for a real val set if available
    'nc'    : 1,
    'names' : ['test_tube'],
}
with open(DATA_YAML, 'w') as fh:
    yaml.dump(data_cfg, fh, default_flow_style=False, sort_keys=False)

# 6. Sanity check
n_imgs = len(list(IMG_TRAIN.iterdir()))
n_lbls = len(list(LBL_TRAIN.iterdir()))
print(f'data.yaml written  -> {DATA_YAML}')
print(f'Images in images/train/ : {n_imgs}')
print(f'Labels in labels/train/ : {n_lbls}')
if n_imgs != n_lbls:
    print(f'  [WARN] counts differ — some images have no label file')
print(open(DATA_YAML).read())


data.yaml written  -> ./data.yaml
Images in images/train/ : 70
Labels in labels/train/ : 70
path: /home/visharads/Zeon/crotation/data
train: images/train
val: images/train
nc: 1
names:
- test_tube



---
## Stage 2 — Train YOLO OBB for Bounding-Box Detection

In [56]:
# ── Delete stale Ultralytics cache files before training ─────────────────────
# Ultralytics caches label scans to disk. If the cache was written when labels
# were missing or empty, it will be reused on every subsequent run, making YOLO
# train on 0 instances even though the label files now exist.
import pathlib as _pl
for _cache in _pl.Path('./data').rglob('*.cache'):
    print(f'Removing stale cache: {_cache}')
    _cache.unlink()
print('Cache cleared.\n')

# ── Hyperparameters ───────────────────────────────────────────────────────────
YOLO_EPOCHS    = 100
YOLO_IMGSZ     = 640
YOLO_BATCH     = 8
YOLO_PATIENCE  = 30   # early-stop if no improvement
YOLO_RUN_NAME  = 'train'

yolo = YOLO(YOLO_PRETRAINED)

yolo.train(
    data      = DATA_YAML,
    epochs    = YOLO_EPOCHS,
    imgsz     = YOLO_IMGSZ,
    batch     = YOLO_BATCH,
    patience  = YOLO_PATIENCE,
    device    = 0 if torch.cuda.is_available() else 'cpu',
    name      = YOLO_RUN_NAME,
    exist_ok  = True,
    fliplr    = 0.5,
    hsv_h     = 0.015,
    hsv_s     = 0.7,
    hsv_v     = 0.4,
    mosaic    = 1.0,
)

# ── Resolve weights path robustly ────────────────────────────────────────────
# yolo.trainer.best is a pathlib.Path Ultralytics sets after training.
# Fallback: glob for the most-recently-modified best.pt under runs/obb/
# so the cell works even when exist_ok=False creates train2, train3, …
import glob as _glob

if hasattr(yolo, 'trainer') and hasattr(yolo.trainer, 'best'):
    YOLO_WEIGHTS = str(yolo.trainer.best)
else:
    _candidates = sorted(
        _glob.glob('./runs/obb/**/best.pt', recursive=True),
        key=os.path.getmtime,
    )
    if not _candidates:
        raise FileNotFoundError(
            'No best.pt found under runs/obb/. '
            'Did training complete successfully?'
        )
    YOLO_WEIGHTS = _candidates[-1]

print(f'\nYOLO training complete.')
print(f'Best weights  -> {YOLO_WEIGHTS}')


Removing stale cache: data/annotated_images/train.cache
Cache cleared.

Ultralytics 8.4.51 🚀 Python-3.13.7 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7806MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=./yolov8n-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=Fal

### (Optional) Quick YOLO inference check

In [57]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection

yolo_inf = YOLO(YOLO_WEIGHTS)

sample_images = [
    os.path.join(IMAGES_DIR, f)
    for f in os.listdir(IMAGES_DIR)
    if f.lower().endswith(('.png', '.jpg', '.jpeg'))
][:3]

fig, axes = plt.subplots(1, len(sample_images), figsize=(6 * len(sample_images), 5))
if len(sample_images) == 1:
    axes = [axes]

for ax, img_path in zip(axes, sample_images):
    bgr   = cv2.imread(img_path)
    rgb   = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    res   = yolo_inf(img_path, verbose=False)[0]
    ax.imshow(rgb)
    if res.obb is not None:
        for obb in res.obb:
            xywhr = obb.xywhr[0].cpu().numpy()
            rect  = ((xywhr[0], xywhr[1]), (xywhr[2], xywhr[3]),
                      math.degrees(xywhr[4]))
            pts   = cv2.boxPoints(rect)
            poly  = Polygon(pts, closed=True, fill=False,
                            edgecolor='lime', linewidth=2)
            ax.add_patch(poly)
    ax.set_title(os.path.basename(img_path))
    ax.axis('off')

plt.tight_layout()
plt.show()

<Figure size 1800x500 with 3 Axes>

---
## Stage 3 — Train ResNet-18 for Angle Estimation

Uses **ground-truth bounding boxes** from `annotations.csv` to crop each tube,  
then regresses `[sin(θ), cos(θ)]` to avoid the 0°/360° discontinuity.

In [58]:
class TubeAngleDataset(Dataset):
    """
    Reads annotations.csv, crops each tube region using the ground-truth bbox,
    and returns (image_tensor, [sin(angle), cos(angle)]).
    """

    def __init__(self, csv_path, images_dir, transform=None):
        self.df        = pd.read_csv(csv_path)
        self.images_dir = images_dir
        self.transform  = transform

        required = {'image', 'center_x', 'center_y', 'bbox_w',
                    'bbox_h', 'bbox_rotation', 'angle_deg'}
        missing  = required - set(self.df.columns)
        if missing:
            raise ValueError(f'annotations.csv missing columns: {missing}')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.images_dir, row['image'])

        crop = crop_rotated_bbox(
            img_path,
            row['center_x'], row['center_y'],
            row['bbox_w'],   row['bbox_h'],
            row['bbox_rotation'],
        )
        crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)  # HWC uint8 RGB

        if self.transform:
            crop = self.transform(crop)

        angle_rad = math.radians(float(row['angle_deg']))
        target    = torch.tensor([math.sin(angle_rad),
                                   math.cos(angle_rad)], dtype=torch.float32)
        return crop, target


# ── Transforms ────────────────────────────────────────────────────────────────
# IMPORTANT: no random flips / geometric rotations — they would corrupt the
# angle label. Only colour / texture augmentations are safe.
train_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std= [0.229, 0.224, 0.225]),
])

inference_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std= [0.229, 0.224, 0.225]),
])

dataset    = TubeAngleDataset(ANNOTATIONS_CSV, IMAGES_DIR, transform=train_transforms)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True,
                        num_workers=0, pin_memory=torch.cuda.is_available())
print(f'Dataset: {len(dataset)} tube instances')

Dataset: 371 tube instances


In [59]:
# ── ResNet-18 with a 2-output regression head ─────────────────────────────────
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
resnet.fc = nn.Linear(resnet.fc.in_features, 2)  # [sin, cos]
resnet    = resnet.to(DEVICE)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(resnet.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

print(f'ResNet-18 built. Output head: {resnet.fc}')

ResNet-18 built. Output head: Linear(in_features=512, out_features=2, bias=True)


In [60]:
RESNET_EPOCHS = 50   # increase if loss is still falling at 50
loss_history  = []

print(f'Training ResNet-18 on {DEVICE} for {RESNET_EPOCHS} epochs …\n')

for epoch in range(1, RESNET_EPOCHS + 1):
    resnet.train()
    running_loss = 0.0

    for images, targets in dataloader:
        images  = images.to(DEVICE)
        targets = targets.to(DEVICE)

        optimizer.zero_grad()
        outputs = resnet(images)
        loss    = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(dataloader)
    loss_history.append(avg_loss)
    scheduler.step(avg_loss)

    if epoch % 5 == 0 or epoch == 1:
        print(f'  Epoch {epoch:>3}/{RESNET_EPOCHS} | Loss: {avg_loss:.5f}')

# Save weights
torch.save(resnet.state_dict(), RESNET_WEIGHTS)
print(f'\nTraining complete. Weights saved → {RESNET_WEIGHTS}')

Training ResNet-18 on cuda for 50 epochs …

  Epoch   1/50 | Loss: 0.80539
  Epoch   5/50 | Loss: 0.26809
  Epoch  10/50 | Loss: 0.10914
  Epoch  15/50 | Loss: 0.06989
  Epoch  20/50 | Loss: 0.03524
  Epoch  25/50 | Loss: 0.03133
  Epoch  30/50 | Loss: 0.01763
  Epoch  35/50 | Loss: 0.02439
  Epoch  40/50 | Loss: 0.02642
  Epoch  45/50 | Loss: 0.00708
  Epoch  50/50 | Loss: 0.00383

Training complete. Weights saved → ./resnet18_angle.pth


In [61]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(loss_history) + 1), loss_history, lw=2)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('ResNet-18 Angle Regression — Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('./resnet_loss_curve.png', dpi=120)
plt.show()

<Figure size 800x400 with 1 Axes>

In [62]:
# ── Evaluate on the training set (sanity check — use a held-out set if possible)
resnet.eval()
eval_transforms = inference_transforms

angle_errors = []
df_ann = pd.read_csv(ANNOTATIONS_CSV)

with torch.no_grad():
    for _, row in df_ann.iterrows():
        img_path = os.path.join(IMAGES_DIR, row['image'])
        try:
            crop = crop_rotated_bbox(img_path,
                                     row['center_x'], row['center_y'],
                                     row['bbox_w'],   row['bbox_h'],
                                     row['bbox_rotation'])
            crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            tensor = eval_transforms(crop).unsqueeze(0).to(DEVICE)

            out = resnet(tensor)[0]
            pred_rad = math.atan2(out[0].item(), out[1].item())
            pred_deg = math.degrees(pred_rad) % 360

            # Circular error (shortest arc between two angles)
            err = abs(pred_deg - row['angle_deg'])
            err = min(err, 360 - err)
            angle_errors.append(err)
        except Exception as e:
            print(f'  [WARN] Skipped {row["image"]}: {e}')

mae  = np.mean(angle_errors)
p90  = np.percentile(angle_errors, 90)
print(f'Train-set evaluation ({len(angle_errors)} tubes)')
print(f'  MAE : {mae:.2f}°')
print(f'  90th: {p90:.2f}°')
print(f'  Max : {max(angle_errors):.2f}°')

Train-set evaluation (371 tubes)
  MAE : 2.85°
  90th: 5.86°
  Max : 9.59°


---
## Stage 4 — Full Inference Pipeline

For every image in `annotated_images/`:  
1. **YOLO** detects tubes → oriented bounding boxes  
2. **crop_rotated_bbox** extracts each tube crop  
3. **ResNet-18** predicts the angle  
4. Results are assembled into a CSV with exactly the same columns as `annotations.csv`

In [63]:
# ── Load both trained models ──────────────────────────────────────────────────
yolo_model = YOLO(YOLO_WEIGHTS)

resnet_inf = models.resnet18(weights=None)
resnet_inf.fc = nn.Linear(resnet_inf.fc.in_features, 2)
resnet_inf.load_state_dict(torch.load(RESNET_WEIGHTS, map_location=DEVICE))
resnet_inf = resnet_inf.to(DEVICE)
resnet_inf.eval()

print('Both models loaded.')

Both models loaded.


In [64]:
YOLO_CONF  = 0.25   # detection confidence threshold
YOLO_IOU   = 0.45   # NMS IoU threshold

final_results = []
image_files   = sorted(
    f for f in os.listdir(IMAGES_DIR)
    if f.lower().endswith(('.png', '.jpg', '.jpeg'))
)

print(f'Running inference on {len(image_files)} images …')

for image_name in image_files:
    image_path  = os.path.join(IMAGES_DIR, image_name)
    yolo_results = yolo_model(image_path, verbose=False,
                              conf=YOLO_CONF, iou=YOLO_IOU)

    for result in yolo_results:
        if result.obb is None or len(result.obb) == 0:
            # No tubes detected — record a placeholder row so the image still
            # appears in the output (useful for debugging)
            final_results.append({
                'image'       : image_name,
                'center_x'    : None,
                'center_y'    : None,
                'bbox_w'      : None,
                'bbox_h'      : None,
                'bbox_rotation': None,
                'angle_deg'   : None,
            })
            continue

        for obb in result.obb:
            xywhr = obb.xywhr[0].cpu().numpy()
            cx, cy, bw, bh = (float(xywhr[0]), float(xywhr[1]),
                               float(xywhr[2]), float(xywhr[3]))
            bbox_rot_rad = float(xywhr[4])
            bbox_rot_deg = math.degrees(bbox_rot_rad)

            # ── Angle prediction ──────────────────────────────────────────────
            angle_deg = None
            try:
                crop   = crop_rotated_bbox(image_path, cx, cy, bw, bh, bbox_rot_deg)
                crop   = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                tensor = inference_transforms(crop).unsqueeze(0).to(DEVICE)

                with torch.no_grad():
                    out       = resnet_inf(tensor)[0]
                    angle_rad = math.atan2(out[0].item(), out[1].item())
                    angle_deg = math.degrees(angle_rad) % 360

            except Exception as e:
                print(f'  [WARN] Crop/angle failed for {image_name}: {e}')

            final_results.append({
                'image'        : image_name,
                'center_x'     : round(cx, 1),
                'center_y'     : round(cy, 1),
                'bbox_w'       : round(bw, 1),
                'bbox_h'       : round(bh, 1),
                'bbox_rotation': round(bbox_rot_deg, 1),
                'angle_deg'    : round(angle_deg, 1) if angle_deg is not None else None,
            })

# ── Export ────────────────────────────────────────────────────────────────────
# Column order matches annotations.csv exactly
COLUMN_ORDER = ['image', 'center_x', 'center_y', 'bbox_w',
                'bbox_h', 'bbox_rotation', 'angle_deg']

df_pred = pd.DataFrame(final_results)[COLUMN_ORDER]
df_pred.to_csv(OUTPUT_CSV, index=False)

detected = df_pred['angle_deg'].notna().sum()
print(f'\nDone. {detected} tubes detected across {len(image_files)} images.')
print(f'Results saved → {OUTPUT_CSV}')
df_pred.head(10)

Running inference on 70 images …

Done. 371 tubes detected across 70 images.
Results saved → ./predictions.csv


,image,center_x,center_y,bbox_w,bbox_h,bbox_rotation,angle_deg
0,043033e6-color.png,348.8,82.6,50.6,50.3,48.2,124.9
1,043033e6-color.png,463.7,85.4,55.0,49.4,58.4,33.8
2,043033e6-color.png,344.6,182.8,45.8,33.7,90.0,351.0
3,043033e6-color.png,460.9,187.5,33.6,46.8,1.2,81.6
4,04f3656c-color.png,429.5,193.2,43.5,28.6,-0.8,190.4
5,04f3656c-color.png,362.4,193.8,39.8,34.1,1.3,219.7
6,04f3656c-color.png,360.6,137.0,30.4,40.5,1.2,66.5
7,04f3656c-color.png,360.9,80.1,37.5,32.7,1.6,318.7
8,04f3656c-color.png,430.2,136.6,30.6,41.8,1.3,245.3
9,04f3656c-color.png,428.5,79.1,29.3,40.9,3.3,110.9


---
## Stage 5 — Compare Predictions vs Ground Truth

In [65]:
import matplotlib.pyplot as plt

df_gt   = pd.read_csv(ANNOTATIONS_CSV)
df_pred = pd.read_csv(OUTPUT_CSV)

# Match on image name (inner join — keeps only images present in both)
# If multiple tubes per image, we match by row order within the same image.
df_gt['_key']   = df_gt.groupby('image').cumcount()
df_pred['_key'] = df_pred.groupby('image').cumcount()
merged = df_gt.merge(df_pred, on=['image', '_key'],
                     suffixes=('_gt', '_pred'))

# Circular angle error
def circular_error(gt, pred):
    diff = np.abs(gt - pred)
    return np.minimum(diff, 360 - diff)

valid  = merged.dropna(subset=['angle_deg_gt', 'angle_deg_pred'])
errors = circular_error(valid['angle_deg_gt'].values,
                         valid['angle_deg_pred'].values)

print(f'Matched {len(valid)} tubes')
print(f'  MAE  : {errors.mean():.2f}°')
print(f'  Median: {np.median(errors):.2f}°')
print(f'  90th : {np.percentile(errors, 90):.2f}°')
print(f'  Max  : {errors.max():.2f}°')

# ── Scatter plot ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.scatter(valid['angle_deg_gt'], valid['angle_deg_pred'],
           alpha=0.6, edgecolors='k', linewidths=0.4)
ax.plot([0, 360], [0, 360], 'r--', lw=1.5, label='Perfect')
ax.set_xlabel('Ground Truth (°)')
ax.set_ylabel('Predicted (°)')
ax.set_title('Angle: GT vs Predicted')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.hist(errors, bins=20, color='steelblue', edgecolor='white')
ax.axvline(errors.mean(), color='red', lw=1.5, label=f'MAE = {errors.mean():.1f}°')
ax.set_xlabel('Circular Error (°)')
ax.set_ylabel('Count')
ax.set_title('Error Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('./eval_plots.png', dpi=120)
plt.show()

Matched 371 tubes
  MAE  : 77.95°
  Median: 75.70°
  90th : 153.90°
  Max  : 179.70°


<Figure size 1200x500 with 2 Axes>

In [66]:
# ── Visualise predictions overlaid on sample images ───────────────────────────
N_SAMPLES = 4
sample_imgs = df_pred['image'].dropna().unique()[:N_SAMPLES]

fig, axes = plt.subplots(1, len(sample_imgs),
                         figsize=(5 * len(sample_imgs), 5))
if len(sample_imgs) == 1:
    axes = [axes]

for ax, img_name in zip(axes, sample_imgs):
    img_path = os.path.join(IMAGES_DIR, img_name)
    bgr      = cv2.imread(img_path)
    rgb      = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    ax.imshow(rgb)

    rows = df_pred[df_pred['image'] == img_name].dropna()
    for _, row in rows.iterrows():
        cx, cy  = row['center_x'], row['center_y']
        bw, bh  = row['bbox_w'],   row['bbox_h']
        rot     = row['bbox_rotation']
        angle   = row['angle_deg']

        # Draw OBB
        rect = ((cx, cy), (bw, bh), rot)
        pts  = cv2.boxPoints(rect).astype(int)
        from matplotlib.patches import Polygon as MPoly
        ax.add_patch(MPoly(pts, closed=True, fill=False,
                           edgecolor='lime', linewidth=2))

        # Draw angle arrow
        arrow_len = max(bw, bh) * 0.5
        ang_rad   = math.radians(angle)
        ax.annotate('', xy=(cx + arrow_len * math.cos(ang_rad),
                             cy - arrow_len * math.sin(ang_rad)),
                    xytext=(cx, cy),
                    arrowprops=dict(arrowstyle='->', color='red', lw=2))
        ax.text(cx, cy - bh // 2 - 6, f'{angle:.1f}°',
                color='yellow', fontsize=8,
                ha='center', va='bottom',
                bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.5))

    ax.set_title(img_name, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('./prediction_overlay.png', dpi=120)
plt.show()

<Figure size 2000x500 with 4 Axes>

---
## Stage 6 — Metrics: Detection (Precision / Recall / F1) + Angle Error

In [67]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

# ── Config ────────────────────────────────────────────────────────────────────
# A predicted bbox 'matches' a ground-truth bbox if their centres are within
# this many pixels. Adjust to suit your image resolution.
CENTRE_MATCH_PX  = 40   # px threshold for a True Positive
ANGLE_CLOSE_DEG  = 15   # degrees threshold used in the per-bin breakdown

df_gt   = pd.read_csv(ANNOTATIONS_CSV)
df_pred = pd.read_csv(OUTPUT_CSV)

# ── Per-image matching (Hungarian algorithm on centre distance) ───────────────
# For each image: match predicted detections to GT boxes, count TP / FP / FN.
# angle_errors_deg is populated only for matched (TP) pairs.

TP_total = FP_total = FN_total = 0
angle_errors_deg = []

all_images = sorted(set(df_gt['image'].unique()) | set(df_pred['image'].dropna().unique()))

for img in all_images:
    gt_rows   = df_gt[df_gt['image'] == img]
    pred_rows = df_pred[(df_pred['image'] == img) & df_pred['angle_deg'].notna()]

    n_gt   = len(gt_rows)
    n_pred = len(pred_rows)

    if n_gt == 0 and n_pred == 0:
        continue
    if n_gt == 0:
        FP_total += n_pred
        continue
    if n_pred == 0:
        FN_total += n_gt
        continue

    # Build centre-distance cost matrix  [n_gt x n_pred]
    gt_xy   = gt_rows[['center_x', 'center_y']].values
    pred_xy = pred_rows[['center_x', 'center_y']].values
    cost    = np.linalg.norm(
        gt_xy[:, None, :] - pred_xy[None, :, :], axis=2
    )  # shape (n_gt, n_pred)

    row_idx, col_idx = linear_sum_assignment(cost)

    matched_gt   = set()
    matched_pred = set()

    for r, c in zip(row_idx, col_idx):
        if cost[r, c] <= CENTRE_MATCH_PX:
            matched_gt.add(r)
            matched_pred.add(c)
            TP_total += 1

            # Circular angle error for this TP pair
            gt_ang   = gt_rows.iloc[r]['angle_deg']
            pred_ang = pred_rows.iloc[c]['angle_deg']
            err      = abs(gt_ang - pred_ang)
            err      = min(err, 360 - err)
            angle_errors_deg.append(err)

    FP_total += n_pred - len(matched_pred)
    FN_total += n_gt   - len(matched_gt)

# ── Detection metrics ─────────────────────────────────────────────────────────
precision = TP_total / (TP_total + FP_total) if (TP_total + FP_total) > 0 else 0.0
recall    = TP_total / (TP_total + FN_total) if (TP_total + FN_total) > 0 else 0.0
f1        = (2 * precision * recall / (precision + recall)
             if (precision + recall) > 0 else 0.0)

# ── Angle metrics (TP pairs only) ─────────────────────────────────────────────
errs = np.array(angle_errors_deg)
mae  = errs.mean()              if len(errs) else float('nan')
p50  = np.median(errs)          if len(errs) else float('nan')
p90  = np.percentile(errs, 90)  if len(errs) else float('nan')
rmse = np.sqrt((errs**2).mean()) if len(errs) else float('nan')
pct_close = (errs < ANGLE_CLOSE_DEG).mean() * 100 if len(errs) else float('nan')

# ── Print summary ─────────────────────────────────────────────────────────────
print('=' * 50)
print('  DETECTION  (centre-dist threshold: {:g} px)'.format(CENTRE_MATCH_PX))
print('=' * 50)
print(f'  TP : {TP_total:>4d}   FP : {FP_total:>4d}   FN : {FN_total:>4d}')
print(f'  Precision : {precision:.4f}')
print(f'  Recall    : {recall:.4f}')
print(f'  F1 Score  : {f1:.4f}')
print()
print('=' * 50)
print('  ANGLE ERROR  (TP pairs only, n={})'.format(len(errs)))
print('=' * 50)
print(f'  MAE    : {mae:.2f} deg')
print(f'  Median : {p50:.2f} deg')
print(f'  RMSE   : {rmse:.2f} deg')
print(f'  P90    : {p90:.2f} deg')
print(f'  % < {ANGLE_CLOSE_DEG} deg : {pct_close:.1f}%')
print('=' * 50)

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Detection bar
ax = axes[0]
bars = ax.bar(['Precision', 'Recall', 'F1'],
              [precision, recall, f1],
              color=['#4C72B0', '#55A868', '#C44E52'],
              edgecolor='white', width=0.5)
ax.set_ylim(0, 1.05)
ax.set_title('Detection Metrics')
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, [precision, recall, f1]):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')

# 2. Angle error histogram
ax = axes[1]
ax.hist(errs, bins=20, color='steelblue', edgecolor='white')
ax.axvline(mae, color='red', lw=2, label=f'MAE = {mae:.1f}°')
ax.axvline(p50, color='orange', lw=2, linestyle='--', label=f'Median = {p50:.1f}°')
ax.set_xlabel('Circular Angle Error (deg)')
ax.set_ylabel('Count')
ax.set_title('Angle Error Distribution')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# 3. Cumulative error curve
ax = axes[2]
sorted_errs = np.sort(errs)
cdf = np.arange(1, len(sorted_errs) + 1) / len(sorted_errs)
ax.plot(sorted_errs, cdf * 100, lw=2, color='steelblue')
ax.axhline(90, color='red', lw=1, linestyle='--', label='90th pct')
ax.axvline(ANGLE_CLOSE_DEG, color='orange', lw=1,
           linestyle='--', label=f'{ANGLE_CLOSE_DEG}° threshold')
ax.set_xlabel('Angle Error (deg)')
ax.set_ylabel('Cumulative %')
ax.set_title('Cumulative Error Curve')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('./metrics_summary.png', dpi=120)
plt.show()


  DETECTION  (centre-dist threshold: 40 px)
  TP :  371   FP :    0   FN :    0
  Precision : 1.0000
  Recall    : 1.0000
  F1 Score  : 1.0000

  ANGLE ERROR  (TP pairs only, n=371)
  MAE    : 32.15 deg
  Median : 6.80 deg
  RMSE   : 56.72 deg
  P90    : 112.70 deg
  % < 15 deg : 62.8%


<Figure size 1500x400 with 3 Axes>